# Effective Potential for Hydrogen Atom (l = 2)

$$
V_{\text{eff}}(r)
= -\frac{1}{4 \pi \epsilon_o} \frac{e^2}{r} + \frac{\hbar^2 l(l+1)}{2mr^2}
$$

$$
E_n
=
-\frac{m Z^2 e^4}{2 \hbar^2 n^2}
$$

Substituting $ \hbar = 1,\; m = 1,\; Z = 1,\; e =\frac{1}{4 \pi \epsilon_o}= 1 $ and $l=2$ (hence $n\geq 3$):

```
Expected : [-0.056, -0.032, -0.028,...]
```
with respect to `n = [3, 4, 5,...]` of $E_n = -\frac{1}{2n^2}$

**Since $x\neq0$ in the $V(x)$. So we take domain of x : `t_span = (1e-6, L)` to avoid singularity**

### Setup

In [6]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

m, hbar = 1.0, 1.0

def V(x):
  l=2
  return -1/x + 0.5 * l * (l+1) / x **2

def func(x, u_vec, E):
  u, du = u_vec
  d2u = (2.0 * m / hbar**2) * (V(x) - E) * u
  return [du, d2u]

def shoot(E): # domain = (1e-6, L)
  sol = solve_ivp(fun=func, t_span=(1e-6, L), y0=[0, 1], method="RK45", args=(E,), atol=1e-8, rtol=1e-8)

  # u_final = u(L)
  u_final = sol.y[0, -1]

  # if |u_final|>1e10, return u_final to np.sign(u_final) * 1e10 to avoid overflow
  return np.sign(u_final) * min(abs(u_final), 1e10)

### Find optimal L
We guess
```
E_min, E_max = -10, 0
```
We defined `find_optimal()` and find $L_{optimal}$ with

```
for L in np.arange(5, 100, 5):
...
```

In [2]:
E_min, E_max = -10, 0

# Defind find_optimalL
def find_optimalL(E_min, E_max, tol=1e-6):
    print(f"{'L':<10} | {'E_root'}")
    print("-" * 25)
    E_prev = 0.0

#  Change L_min, L_max, and nsteps at here eg. Lmin = 2, Lmax=10, nsteps = 1
    for L in np.arange(5, 100, 5):

      def shoot(E): # domain = (1e-6, L)
          sol = solve_ivp(fun=func, t_span=(1e-6, L), y0=[0, 1], method="RK45", args=(E,), atol=1e-8, rtol=1e-8)

          u_final = sol.y[0, -1]

          return np.sign(u_final) * min(abs(u_final), 1e10)

      energies = np.linspace(E_min, E_max, 100)
      eigenvalues=[]
      f_old = shoot(energies[0])

      for i in range(len(energies) - 1):
          E_low = energies[i]
          E_high = energies[i+1]
          f_new = shoot(E_high)

          if f_old * f_new < 0:
              try:
                  E_root = brentq(shoot, E_low, E_high)
                  eigenvalues.append(E_root)
                  break
              except ValueError:
                  pass

          f_old = f_new

      if not eigenvalues:
          print(f"{L:<10.1f} | No eigenvalues found")
          continue

      E_root = eigenvalues[0]
      print(f"{L:<10.1f} | {E_root:.8f}")

      if abs(E_root - E_prev) < tol:
          print("-" * 25)
          print(f"Stability reached at L = {L:.1f}")

          return L, E_root
      E_prev = E_root

# Execute to find optimal L
optimal_L, stable_E = find_optimalL(E_min, E_max, tol=1e-6)

L          | E_root
-------------------------
5.0        | No eigenvalues found
10.0       | -0.00709278
15.0       | -0.04664258
20.0       | No eigenvalues found
25.0       | No eigenvalues found
30.0       | No eigenvalues found
35.0       | -0.05555290
40.0       | -0.05555533
45.0       | -0.05555554
-------------------------
Stability reached at L = 45.0


> Output : `L = 45.0 and E0 > -0.06`

Thence, we define and run `find_eigenvalues()` with L = 45

We modify `E_min, E_max = -0.06, 0` to speed up

In [4]:
L = 45.0
E_min, E_max = -0.06, 0

# Define find_eigenvalues()
def find_eigenvalues(E_min, E_max, n, shoot=shoot):
    energies = np.linspace(E_min, E_max, n)
    eigenvalues = []

    f_old = shoot(energies[0])

    for i in range(len(energies) - 1):
        E_low = energies[i]
        E_high = energies[i+1]
        f_new = shoot(E_high)

        '''Sign change detected -> root found in this interval'''
        if f_old * f_new < 0:
            E_root = brentq(shoot, E_low, E_high)
            eigenvalues.append(E_root)

        f_old = f_new

    print(eigenvalues)

    if not eigenvalues:
        print("No eigenvalues found. Try different L or E_min, E_max.")

# Execute find_eigenvalues()
find_eigenvalues(E_min, E_max, n=100)

[-0.05555553891597539, -0.031174611130845777, -0.017190161808121465]


When `L = 45.0` the eigenvalues give
```
[-0.05555553891597539, -0.031174611130845777, -0.017190161808121465]
```
Highly aligns what we expected : [-0.056, -0.032, -0.028,...]
